In [31]:
import numpy as np
import pandas as pd
import json

from itertools import product

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier 

In [64]:
#fixing extra params for XGB and Random forest
FIXED_PARAMS = {'n_jobs': -1, 'random_state': 42}

In [32]:
def temporal_split_with_gap(df, min_date, max_date, gap_size=0, train_prop=0.7, val_prop=0.15, test_prop=0.15):
    """
    Splits a DataFrame into three buckets based on the temporal order of the 'Date' column.
    Also is adding 

    Args:
        df (DataFrame): The DataFrame to split.
        min_date (str or Timestamp): Minimum date in the DataFrame.
        max_date (str or Timestamp): Maximum date in the DataFrame.
        gap_size (int) - how many days gap betweeen sets parts
        train_prop (float): Proportion of data for training set (default: 0.6).
        val_prop (float): Proportion of data for validation set (default: 0.2).
        test_prop (float): Proportion of data for test set (default: 0.2).

    Returns:
        DataFrame: The input DataFrame with a new column 'split' indicating the split for each row.
    """
    # Define the date intervals
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)

    mask = (df['date'] <= train_end - pd.Timedelta(days=gap_size))
    df.loc[mask, 'split'] = 'train'
    mask = ((df['date'] > train_end) & (df['date'] <= val_end - pd.Timedelta(days=gap_size)))
    df.loc[mask, 'split'] = 'validation'
    mask = (df['date'] > val_end)
    df.loc[mask, 'split'] = 'test'
    #last 30 days papers
    mask = ((df['is_positive_growth_30d_future'].isna()) & (df['split'] == 'test'))
    df.loc[mask, 'split'] = 'latest'
    
    return df

**Preparing data**

Downloading all features

In [159]:
df_features = pd.read_parquet('../data/features.parquet')

with open('../data/feature_set.json', encoding='utf-8') as f:
    feature_sets = json.load(f)
TO_PREDICT = feature_sets['TO_PREDICT']
NUMERICAL = feature_sets['NUMERICAL']
DUMMIES = feature_sets['DUMMIES']
FEATURES_LIST = NUMERICAL + DUMMIES
#features without macro and month
macro_columns = ['gr_10y', 'de_10y', 'spread_gr_de', 'deposit_rate']
FEATURES_B = [col for col in X_train.columns if col not in macro_columns and not col.startswith('month_')]

Splitting datasets

In [34]:
df_full = df_features.copy()
min_date_df = df_full.date.min()
max_date_df = df_full.date.max()

df_full = temporal_split_with_gap(df_full,
                                 min_date = min_date_df,
                                 max_date = max_date_df,
                                 gap_size = 45)
len(df_full[df_full['split'] == 'train'])

59917

In [35]:
#preparing X sets
X_train = df_full[NUMERICAL + DUMMIES][df_full.split == 'train'].copy()
X_val = df_full[NUMERICAL + DUMMIES][df_full.split == 'validation'].copy()
X_test = df_full[NUMERICAL + DUMMIES][df_full.split == 'test'].copy()

#preparing y sets
y_train = df_full['is_positive_growth_30d_future'][df_full.split == 'train'].copy()
y_val = df_full['is_positive_growth_30d_future'][df_full.split == 'validation'].copy()
y_test = df_full['is_positive_growth_30d_future'][df_full.split == 'test'].copy()

In [36]:
y_train.isna().sum()

np.int64(0)

In [37]:
print(f'Base rate train set {round(y_train.mean(),2)}')
print(f'Base rate validation set {round(y_val.mean(),2)}')
print(f'Base rate test set {round(y_test.mean(),2)}')

Base rate train set 0.57
Base rate validation set 0.58
Base rate test set 0.63


**Models**

In [41]:
def create_model(model_class, params, fixed_params):

    model = model_class(**params, **fixed_params)
    return model

In [78]:
def train_model(model_class, features_list, params, fixed_params=None):
    if fixed_params is None:
        fixed_params = {}
    model = create_model(model_class, params, fixed_params)
    model.fit(X_train[features_list], y_train.astype(int))
    
    return model

In [98]:
def predict_model(model_class, features_list, params, fixed_params=None):
    if fixed_params is None:
        fixed_params = {}
    model = train_model(model_class, features_list, params, fixed_params)
    y_pred = model.predict_proba(X_val[features_list])[:, 1]
    
    return y_pred

In [63]:
def evaluate_model(model_class, features_list, params, proba = 0.5, roc_auc=False, mean=False, precision=False, fixed_params=None):
    """Train model on given features and return its parameters and AUC, mean and precision on validation.""" 
    if fixed_params is None:
        fixed_params = {}
    y_pred = predict_model(model_class, features_list, params, fixed_params)
    y_pred_class = (y_pred >= proba).astype(int)
    if roc_auc:
        params['roc_auc'] = roc_auc_score(y_val, y_pred)
    if mean:
        params['mean'] = y_pred_class.mean()
    if precision:
        params['precision'] = precision_score(y_val, y_pred_class)
    return params

*Decision tree*

In [99]:
y_pred = predict_model(DecisionTreeClassifier, FEATURES_LIST, {})
roc_auc_score(y_val, y_pred)

0.5063105121343272

In [62]:
grid = {'min_samples_leaf': [200], 'max_depth': list(range(2,16))}
proba = 0.8
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(DecisionTreeClassifier, FEATURES_LIST, params, proba, True, True, True))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()
print(df_results.to_string())

    min_samples_leaf  max_depth   roc_auc      mean  precision
2                200          4  0.544202  0.760834   0.612206
3                200          5  0.525253  0.596448   0.591915
4                200          6  0.516445  0.596448   0.591915
5                200          7  0.513724  0.596448   0.591915
6                200          8  0.513510  0.596448   0.591915
7                200          9  0.513510  0.596448   0.591915
8                200         10  0.513510  0.596448   0.591915
9                200         11  0.513510  0.596448   0.591915
10               200         12  0.513510  0.596448   0.591915
11               200         13  0.513510  0.596448   0.591915
12               200         14  0.513510  0.596448   0.591915
13               200         15  0.513510  0.596448   0.591915
1                200          3  0.511326  0.924731   0.589500
0                200          2  0.510650  0.924731   0.589500


Best depth is 4: AUC 0.544, precision 0.612 vs validation base rate 0.58. Deeper trees overfit (AUC goes down), from depth 8 tree stops growing because of min_samples_leaf=200. Improvement is small, but it's typical for stocks. 

In [162]:
grid = {'min_samples_leaf': [200], 'max_depth': list(range(2,16))}
proba = 0.5
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(DecisionTreeClassifier, FEATURES_B, params, proba, True, True, True))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()
print(df_results.to_string())

    min_samples_leaf  max_depth   roc_auc      mean  precision
4                200          6  0.505898  0.872515   0.567361
1                200          3  0.495583  0.642392   0.593710
2                200          4  0.490025  0.989247   0.581769
3                200          5  0.488555  0.917644   0.580293
13               200         15  0.484486  0.651271   0.584740
11               200         13  0.480655  0.721245   0.587757
12               200         14  0.480342  0.663490   0.586618
0                200          2  0.473608  0.999756   0.583883
10               200         12  0.467008  0.667889   0.574704
5                200          7  0.466796  0.921065   0.569116
6                200          8  0.466213  0.886445   0.572229
8                200         10  0.466021  0.725236   0.574525
9                200         11  0.464927  0.656403   0.570489
7                200          9  0.458172  0.883757   0.571481


Decision Tree without macro and month doesn't find anything: best AUC is 0.506 (depth 6), it's random level. For many depths AUC is below 0.5, so rules found on train work the opposite way on validation — the tree learns noise. Stock-level signal is weak, one tree can't catch it, Random Forest and XGBoost do better because they average many trees. 

*Random forest*

In [100]:
grid = {'min_samples_leaf': [200], 'n_estimators': [50, 100, 200, 500, 1000, 1500, 2000]}
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(RandomForestClassifier, FEATURES_LIST, params, proba, True, True, True, FIXED_PARAMS))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()
print(df_results.to_string())

   min_samples_leaf  n_estimators   roc_auc      mean  precision
6               200          2000  0.607540  0.818019   0.621091
5               200          1500  0.605386  0.826002   0.620611
4               200          1000  0.604471  0.820463   0.621326
3               200           500  0.601234  0.843353   0.616633
2               200           200  0.581135  0.828853   0.614742
1               200           100  0.578696  0.827794   0.611494
0               200            50  0.555015  0.769225   0.605528


We can see that AUC growth from 50 to 500 trees, after 500 it is plateau so we will leave only from 100 to 1000 for testing max_depth

In [71]:
grid = {'min_samples_leaf': [200], 'n_estimators': [100, 200, 500, 1000], 'max_depth': list(range(2,16))}
proba = 0.5
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(RandomForestClassifier, FEATURES_LIST, params, proba, True, True, True, fixed_params=FIXED_PARAMS))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()
print(df_results.to_string())

    min_samples_leaf  n_estimators  max_depth   roc_auc  mean  precision
42               200          1000          2  0.638281   1.0   0.583904
1                200           100          3  0.636422   1.0   0.583904
15               200           200          3  0.634279   1.0   0.583904
43               200          1000          3  0.631657   1.0   0.583904
2                200           100          4  0.631562   1.0   0.583904
29               200           500          3  0.630153   1.0   0.583904
28               200           500          2  0.625931   1.0   0.583904
44               200          1000          4  0.625432   1.0   0.583904
46               200          1000          6  0.625077   1.0   0.583904
30               200           500          4  0.622010   1.0   0.583904
32               200           500          6  0.621593   1.0   0.583904
14               200           200          2  0.621500   1.0   0.583904
45               200          1000          5  0.62

In [91]:
params = {'n_estimators': 100, 'max_depth': 3, 'min_samples_leaf': 200}

rf = train_model(RandomForestClassifier, FEATURES_LIST, params, FIXED_PARAMS)
y_pred = rf.predict_proba(X_val[FEATURES_LIST])[:, 1]
    
pd.Series(y_pred).describe()

count    12276.000000
mean         0.628029
std          0.020421
min          0.570184
25%          0.617889
50%          0.627973
75%          0.637033
max          0.678258
dtype: float64

In [163]:
#test b
grid = {'min_samples_leaf': [200], 'n_estimators': [100, 200, 500, 1000], 'max_depth': list(range(2,16))}
proba = 0.5
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(RandomForestClassifier, FEATURES_B, params, proba, True, True, True, fixed_params=FIXED_PARAMS))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()
print(df_results.to_string())

    min_samples_leaf  n_estimators  max_depth   roc_auc      mean  precision
47               200          1000          7  0.548879  0.985663   0.586942
33               200           500          7  0.548580  0.986559   0.587482
17               200           200          5  0.548346  1.000000   0.583904
31               200           500          5  0.547846  1.000000   0.583904
45               200          1000          5  0.546936  1.000000   0.583904
32               200           500          6  0.546147  0.993239   0.585828
12               200           100         14  0.546035  0.860459   0.593676
48               200          1000          8  0.545306  0.970512   0.588299
46               200          1000          6  0.544915  0.993320   0.585534
18               200           200          6  0.544811  0.995112   0.584561
49               200          1000          9  0.544582  0.946807   0.590811
34               200           500          8  0.544109  0.967579   0.589072

Random Forest without macro and month: tuning improved AUC from 0.537 (parameters from model A) to 0.548 (200 trees, depth 5). Like XGBoost B, the best depth is bigger than for model A: without macro the model needs more splits to use stock-level features. Top results (1000/7, 500/7, 200/5) differ less than 0.001, so we take the fastest one.


In [164]:
params = {'n_estimators': 200, 'max_depth': 5, 'min_samples_leaf': 200}

rf = train_model(RandomForestClassifier, FEATURES_LIST, params, FIXED_PARAMS)
y_pred = rf.predict_proba(X_val[FEATURES_LIST])[:, 1]
    
pd.Series(y_pred).describe()

count    12276.000000
mean         0.665713
std          0.026570
min          0.570139
25%          0.650601
50%          0.669451
75%          0.684191
max          0.726100
dtype: float64

In [81]:
proba_set = [0.62, 0.63, 0.64, 0.65, 0.66]
for proba in proba_set:
    y_pred_class = (y_pred >= proba).astype(int)
    precision = precision_score(y_val, y_pred_class)
    print(f'proba={proba} precision={precision}')

proba=0.62 precision=0.635552039778506
proba=0.63 precision=0.6786438179473006
proba=0.64 precision=0.8060721062618595
proba=0.65 precision=0.8253706754530478
proba=0.66 precision=0.846830985915493


In [86]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False).head(15)

gr_10y            0.147223
de_10y            0.100634
deposit_rate      0.097395
spread_gr_de      0.086363
month_8           0.056282
month_12          0.041819
month_3           0.036439
month_11          0.035779
daily_turnover    0.032089
month_9           0.025847
month_2           0.023396
trix              0.023326
roc               0.018623
natr              0.017749
growth_252d       0.016934
dtype: float64

In [87]:
def presicion_quantile(quantile_set, y_pred, y_val):
    for q in quantile_set:
        proba = pd.Series(y_pred).quantile(q)
        y_pred_class = (y_pred >= proba).astype(int)
        precision = precision_score(y_val, y_pred_class)
        print(f'For quantile={q} proba={proba} mean={y_pred_class.mean()} precision_score={precision}')

In [160]:
features_b = FEATURES_B
#the same forest as the best one
params = {'min_samples_leaf': 200, 'n_estimators': 100, 'max_depth': 3}
y_pred = predict_model(RandomForestClassifier, features_b, params, FIXED_PARAMS)

roc_auc = roc_auc_score(y_val, y_pred)
print(f'roc_auc_score without macro and month={roc_auc}')


#thresholds by quantiles: top 50%, 25%, 10%
quantile_set = [0.5, 0.75, 0.9]

presicion_quantile(quantile_set, y_pred, y_val)

roc_auc_score without macro and month=0.5371762753759859
For quantile=0.5 proba=0.5633557931961363 mean=0.5004072987943956 precision_score=0.5993814097346574
For quantile=0.75 proba=0.5674104941019774 mean=0.25008145975887913 precision_score=0.6299674267100978
For quantile=0.9 proba=0.5738755920316222 mean=0.10019550342130987 precision_score=0.6560975609756098


Model A (all features): AUC 0.638, most important are macro and month, so it predicts good periods for the whole market, not good stocks. Model B (without macro and month): AUC 0.537, precision grows with threshold: 0.60 → 0.63 → 0.66 vs base rate 0.584. Both go to test, and both can be used as two strategies in simulation. 


In [112]:
features_a = FEATURES_LIST
grid = {'n_estimators': [300, 400, 500], 'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.3]}
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(XGBClassifier, features_a, params, roc_auc=True, fixed_params=FIXED_PARAMS))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()

,n_estimators,learning_rate,roc_auc
1,300,0.01,0.564506
6,400,0.01,0.563404
11,500,0.01,0.559379
2,300,0.05,0.556309
3,300,0.10,0.556218


Best learning_rate is 0.01

In [114]:
grid = {'n_estimators': [300, 400, 500], 'learning_rate': [0.01], 'max_depth': list(range(2,9))}
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(XGBClassifier, features_a, params, roc_auc=True, fixed_params=FIXED_PARAMS))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()

,n_estimators,learning_rate,max_depth,roc_auc
7,400,0.01,2,0.607255
0,300,0.01,2,0.605486
14,500,0.01,2,0.593844
17,500,0.01,5,0.569415
3,300,0.01,5,0.567952


Best max_depth is 2 and 5

In [115]:
grid = {'n_estimators': [300, 400, 500], 'learning_rate': [0.01], 'max_depth': [2,5], 'min_child_weight': [1, 10, 30]}
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(XGBClassifier, features_a, params, roc_auc=True, fixed_params=FIXED_PARAMS))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()

,n_estimators,learning_rate,max_depth,min_child_weight,roc_auc
8,400,0.01,2,30,0.609807
2,300,0.01,2,30,0.608226
6,400,0.01,2,1,0.607255
7,400,0.01,2,10,0.607255
0,300,0.01,2,1,0.605486


Because again won the small depth its meaning the best decisions was made on macro economical features. lets test it on model B (no macroeconomic features)

In [116]:
grid = {'n_estimators': [300, 400, 500], 'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.3]}
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(XGBClassifier, features_b, params, roc_auc=True, fixed_params=FIXED_PARAMS))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()

,n_estimators,learning_rate,roc_auc
12,500,0.05,0.531460
2,300,0.05,0.528791
3,300,0.10,0.527155
7,400,0.05,0.526160
8,400,0.10,0.525785


In [118]:
grid = {'n_estimators': [300, 400, 500], 'learning_rate': [0.05], 'max_depth': list(range(2,9))}
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(XGBClassifier, features_b, params, roc_auc=True, fixed_params=FIXED_PARAMS))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()

,n_estimators,learning_rate,max_depth,roc_auc
5,300,0.05,7,0.546070
12,400,0.05,7,0.545838
19,500,0.05,7,0.542922
17,500,0.05,5,0.540232
10,400,0.05,5,0.535769


In [120]:
grid = {'n_estimators': [300, 400, 500], 'learning_rate': [0.05], 'max_depth': [5, 7], 'min_child_weight': [0.25, 0.5, 1, 10, 30]}
results = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    results.append(evaluate_model(XGBClassifier, features_b, params, roc_auc=True, fixed_params=FIXED_PARAMS))
df_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
df_results.head()

,n_estimators,learning_rate,max_depth,min_child_weight,roc_auc
7,300,0.05,7,1.0,0.546070
17,400,0.05,7,1.0,0.545838
27,500,0.05,7,1.0,0.542922
18,400,0.05,7,10.0,0.541818
28,500,0.05,7,10.0,0.540902


Depth 7 learning rate 0.05 with auc 0.546 is the best model B. Going to the next stage

Precision and mean on quantiles for XGBoost A and B on validation

In [121]:
#thresholds by quantiles: top 50%, 25%, 10%
quantile_set = [0.5, 0.75, 0.9]

#model_A 
params = {'n_estimators': 400, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 30}
y_pred = predict_model(XGBClassifier, features_a, params, FIXED_PARAMS)
print("Full features")
presicion_quantile(quantile_set, y_pred, y_val)

#model_B
params = {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 1}
y_pred = predict_model(XGBClassifier, features_b, params, FIXED_PARAMS)
print("Only local features")
presicion_quantile(quantile_set, y_pred, y_val)

Full features
For quantile=0.5 proba=0.7645977139472961 mean=0.5010589768654284 precision_score=0.6450983579905707
For quantile=0.75 proba=0.7880019545555115 mean=0.25162919517758225 precision_score=0.7552606021366138
For quantile=0.9 proba=0.805162250995636 mean=0.10475724991854024 precision_score=0.8203732503888025
Only local features
For quantile=0.5 proba=0.5464612245559692 mean=0.5 precision_score=0.6177908113391984
For quantile=0.75 proba=0.6692663729190826 mean=0.25 precision_score=0.649723036819811
For quantile=0.9 proba=0.7664461731910706 mean=0.10003258390355164 precision_score=0.6710097719869706


XGBoost shows the same picture as Random Forest. Model A (all features): precision 0.65 → 0.76 → 0.82 for top 50% / 25% / 10% vs base rate 0.584. It is strong again because of macro regime. Model B (without macro and month): precision 0.62 → 0.65 → 0.67, a bit better than Random Forest B (0.60 → 0.63 → 0.66). Final check will be on test set (2025–2026). 

*Final evaluation on the test set*

In [170]:
def quantil_proba(quantile_set, y_pred_val, y_pred_test, y_val, y_test, result):
    for key, value in quantile_set.items():
        proba = pd.Series(y_pred_val).quantile(value)
        result['treshold' + key] = proba
        y_pred_class_val = (y_pred_val >= proba).astype(int)
        precision_val = precision_score(y_val, y_pred_class_val, zero_division=0)
        result['precision_val' + key] = precision_val
        y_pred_class_test = (y_pred_test >= proba).astype(int)
        precision_test = precision_score(y_test, y_pred_class_test, zero_division=0)
        result['precision_test' + key] = precision_test
        result['signal_test' + key] = y_pred_class_test.mean()
    return result

In [171]:
def evalute_result(model_class, features, quantile_set, base_rate_test, params, fixed_params=None):
    if fixed_params is None:
        fixed_params = {}
    features_name, features_list = features
    model = train_model(model_class, features_list, params, fixed_params)
    y_pred_val = model.predict_proba(X_val[features_list])[:, 1]
    y_pred_test = model.predict_proba(X_test[features_list])[:, 1]
    result = {'model': model_class.__name__, 
              'features': features_name, 
              'auc_val':roc_auc_score(y_val, y_pred_val), 
              'auc_test':roc_auc_score(y_test, y_pred_test),
              'base_rate_test': base_rate_test
             }
    result = quantil_proba(quantile_set, y_pred_val, y_pred_test, y_val, y_test, result)
    return result

In [172]:
quantile_set = {'_top25': 0.75, '_top10': 0.9}
features_all_set = ('all', features_a)
features_no_macro_set = ('no_macro', features_b)
results = []
base_rate_test = y_test.mean()

best_configs = [
    (DecisionTreeClassifier, features_all_set,
     {'max_depth': 4, 'min_samples_leaf': 200}, None),
    (DecisionTreeClassifier, features_no_macro_set,
     {'max_depth': 6, 'min_samples_leaf': 200}, None),
    (RandomForestClassifier, features_all_set,
     {'n_estimators': 100, 'max_depth': 3, 'min_samples_leaf': 200}, FIXED_PARAMS),
    (RandomForestClassifier, features_no_macro_set,
     {'n_estimators': 200, 'max_depth': 5, 'min_samples_leaf': 200}, FIXED_PARAMS),
    (XGBClassifier, features_all_set,
     {'n_estimators': 400, 'max_depth': 2, 'learning_rate': 0.01, 'min_child_weight': 30}, FIXED_PARAMS),
    (XGBClassifier, features_no_macro_set,
     {'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.05, 'min_child_weight': 1}, FIXED_PARAMS),
]
for model_class, features, params, fixed_params in best_configs:
    result = evalute_result(model_class, features, quantile_set, base_rate_test, params, fixed_params)
    results.append(result)
df_results = pd.DataFrame(results)
df_results.head(6)

,model,features,auc_val,auc_test,base_rate_test,treshold_top25,precision_val_top25,precision_test_top25,signal_test_top25,treshold_top10,precision_val_top10,precision_test_top10,signal_test_top10
0,DecisionTreeClassifier,all,0.544202,0.539805,0.631162,0.925321,0.612206,0.000000,0.000000,0.925321,0.612206,0.000000,0.000000
1,DecisionTreeClassifier,no_macro,0.505898,0.494031,0.631162,0.586360,0.600329,0.633052,0.594634,0.586360,0.600329,0.633052,0.594634
2,RandomForestClassifier,all,0.636422,0.659622,0.631162,0.637033,0.778429,0.687375,0.080653,0.659009,0.844589,0.589404,0.012203
3,RandomForestClassifier,no_macro,0.548346,0.530440,0.631162,0.566852,0.640925,0.671026,0.197753,0.578474,0.692182,0.654633,0.067157
4,XGBClassifier,all,0.609807,0.657125,0.631162,0.788002,0.755261,0.000000,0.000000,0.805162,0.820373,0.000000,0.000000
5,XGBClassifier,no_macro,0.546070,0.531739,0.631162,0.669266,0.649723,0.699303,0.208825,0.766446,0.671010,0.703429,0.087199


On test, models with macro (A) still rank well (AUC 0.66), but their probabilities moved down: thresholds from validation almost don't work (XGBoost A — no signals, Random Forest A — 8% and 1%). The big gain they had on validation (+17–26 points over base rate) disappears. Models without macro (B) are stable: XGBoost B gives +7 points over base rate on both validation and test (precision 0.70 vs 0.631), and signal share stays close to 25% / 10%. XGBoost B is the best model for simulation. Decision Tree doesn't work in any version.

*Save models for simulation*

In [173]:
#models for simulation: Random Forest and XGBoost without macro
sim_configs = {'rf_b': best_configs[3], 'xgb_b': best_configs[5]}


#rows for simulation: test and latest
df_pred = df_full[df_full['split'].isin(['test', 'latest'])][
    ['date', 'ticker', 'open', 'adj_close', 'growth_future_30d',
     'is_positive_growth_30d_future', 'split']].copy()


for name, (model_class, features, params, fixed_params) in sim_configs.items():
    features_name, features_list = features
    model = train_model(model_class, features_list, params, fixed_params)


    #thresholds from validation
    y_pred_val = model.predict_proba(X_val[features_list])[:, 1]
    threshold_top25 = np.quantile(y_pred_val, 0.75)
    threshold_top10 = np.quantile(y_pred_val, 0.9)


    #probability and signals for test and latest
    df_pred[f'proba_{name}'] = model.predict_proba(df_full.loc[df_pred.index, features_list])[:, 1]
    df_pred[f'signal_{name}_top25'] = (df_pred[f'proba_{name}'] >= threshold_top25).astype(int)
    df_pred[f'signal_{name}_top10'] = (df_pred[f'proba_{name}'] >= threshold_top10).astype(int)


df_pred.to_parquet('../data/predictions.parquet', index=False)
